# Renewable Energy Local Spark Notebook

It uses a local Spark session, reads this repo's ENTSO-E XML and ERA5 NetCDF sample files from `bronze/`, and writes local parquet outputs under `silver/notebook/`.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'spark').exists():
            return candidate
    raise RuntimeError('Could not find repo root from current working directory')

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

ROOT

In [ ]:
country = 'GR'
sample_date = '2025-03-01'

entsoe_files = sorted((ROOT / f'bronze/entsoe/{country}').glob('generation/**/*.xml'))
era5_files = sorted((ROOT / f'bronze/era5/{country}').glob(f'**/{sample_date}_*.nc'))

print('ENTSO-E files:', *entsoe_files, sep='\n- ')
print()
print('ERA5 files:', *era5_files, sep='\n- ')

## If local bronze files are missing

Sync the sample day from GCS first, then rerun the notebook cells:

```bash
mkdir -p bronze/entsoe/GR/generation/2025-03-01 bronze/era5/GR/2025/03
gcloud storage cp "gs://$GCS_BUCKET/bronze/entsoe/GR/generation/2025-03-01/generation.xml" bronze/entsoe/GR/generation/2025-03-01/generation.xml
gcloud storage cp "gs://$GCS_BUCKET/bronze/era5/GR/2025/03/2025-03-01_*.nc" bronze/era5/GR/2025/03/
```

In [3]:
if not entsoe_files:
    raise FileNotFoundError('No local ENTSO-E XML files found under bronze/entsoe. Sync the sample file from GCS first.')

if not era5_files:
    raise FileNotFoundError('No local ERA5 NetCDF files found under bronze/era5. Sync the sample files from GCS first.')

In [4]:
import pandas as pd
from lxml import etree
from pyspark.sql import SparkSession, functions as F, types as T

from spark.utils.era5_netcdf_helpers import netcdf_to_dataframe

In [ ]:
spark = SparkSession.builder \
    .master('local[2]') \
    .appName('renewable-energy-local-spark') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.default.parallelism', '4') \
    .getOrCreate()

spark.version

In [6]:
SOURCE_MAP = {
    'B01': 'Biomass',
    'B02': 'Fossil Brown coal/Lignite',
    'B03': 'Fossil Coal-derived gas',
    'B04': 'Fossil Gas',
    'B05': 'Fossil Hard coal',
    'B06': 'Fossil Oil',
    'B07': 'Fossil Oil shale',
    'B08': 'Fossil Peat',
    'B09': 'Geothermal',
    'B10': 'Hydro Pumped Storage',
    'B11': 'Hydro Run-of-river and poundage',
    'B12': 'Hydro Water Reservoir',
    'B13': 'Marine',
    'B14': 'Nuclear',
    'B15': 'Other renewable',
    'B16': 'Solar',
    'B17': 'Waste',
    'B18': 'Wind Offshore',
    'B19': 'Wind Onshore',
    'B20': 'Other',
}

def resolve_source(code: str) -> str:
    return SOURCE_MAP.get(code, f'Unknown ({code})')

def parse_generation_xml(xml_path: Path, country: str) -> pd.DataFrame:
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    rows = []

    for ts in root.xpath(".//*[local-name()='TimeSeries']"):
        psr_type = ts.xpath(".//*[local-name()='MktPSRType']/*[local-name()='psrType']")
        psr_code = (psr_type[0].text if psr_type else 'unknown') or 'unknown'
        energy_source = resolve_source(psr_code)

        for period in ts.xpath(".//*[local-name()='Period']"):
            start_el = period.xpath(".//*[local-name()='timeInterval']/*[local-name()='start']")
            resolution_el = period.xpath(".//*[local-name()='resolution']")
            if not start_el:
                continue

            start_ts = pd.Timestamp(start_el[0].text)
            resolution = resolution_el[0].text if resolution_el else 'PT60M'
            if '15M' in resolution:
                step_minutes = 15
            elif '30M' in resolution:
                step_minutes = 30
            else:
                step_minutes = 60

            for point in period.xpath(".//*[local-name()='Point']"):
                position = point.xpath(".//*[local-name()='position']")
                quantity = point.xpath(".//*[local-name()='quantity']")
                if not position or not quantity:
                    continue

                try:
                    pos = int(position[0].text)
                    qty = float(quantity[0].text)
                except (TypeError, ValueError):
                    continue

                rows.append({
                    'timestamp': start_ts + pd.Timedelta(minutes=step_minutes * (pos - 1)),
                    'country': country,
                    'energy_source': energy_source,
                    'psr_type': psr_code,
                    'actual_MW': qty,
                })

    return pd.DataFrame(rows)


In [7]:
entsoe_pdf = pd.concat([parse_generation_xml(path, country) for path in entsoe_files], ignore_index=True)
entsoe_pdf['timestamp'] = pd.to_datetime(entsoe_pdf['timestamp'], errors='coerce', utc=True).dt.tz_localize(None)
entsoe_pdf['actual_MW'] = pd.to_numeric(entsoe_pdf['actual_MW'], errors='coerce')
entsoe_pdf = entsoe_pdf.dropna(subset=['timestamp'])
entsoe_pdf['timestamp'] = entsoe_pdf['timestamp'].apply(lambda ts: ts.to_pydatetime())
entsoe_pdf.head()

,timestamp,country,energy_source,psr_type,actual_MW
0,2025-03-01 00:00:00,GR,Fossil Brown coal/Lignite,B02,493.0
1,2025-03-01 01:00:00,GR,Fossil Brown coal/Lignite,B02,501.0
2,2025-03-01 02:00:00,GR,Fossil Brown coal/Lignite,B02,504.0
3,2025-03-01 03:00:00,GR,Fossil Brown coal/Lignite,B02,508.0
4,2025-03-01 04:00:00,GR,Fossil Brown coal/Lignite,B02,506.0


In [8]:
entsoe_schema = T.StructType([
    T.StructField('timestamp', T.TimestampType(), True),
    T.StructField('country', T.StringType(), True),
    T.StructField('energy_source', T.StringType(), True),
    T.StructField('psr_type', T.StringType(), True),
    T.StructField('actual_MW', T.DoubleType(), True)
])

entsoe_records = []
for row in entsoe_pdf.to_dict('records'):
    ts = row['timestamp']
    if hasattr(ts, 'to_pydatetime'):
        ts = ts.to_pydatetime()
    entsoe_records.append({
        'timestamp': ts,
        'country': row['country'],
        'energy_source': row['energy_source'],
        'psr_type': row['psr_type'],
        'actual_MW': float(row['actual_MW']) if row['actual_MW'] is not None else None,
    })

entsoe_sdf = spark.createDataFrame(entsoe_records, schema=entsoe_schema)
entsoe_sdf.printSchema()
entsoe_sdf.show(5, truncate=False)

root
 |-- timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- energy_source: string (nullable = true)
 |-- psr_type: string (nullable = true)
 |-- actual_MW: double (nullable = true)



[Stage 0:>                                                          (0 + 1) / 1]

+-------------------+-------+-------------------------+--------+---------+
|timestamp          |country|energy_source            |psr_type|actual_MW|
+-------------------+-------+-------------------------+--------+---------+
|2025-03-01 00:00:00|GR     |Fossil Brown coal/Lignite|B02     |493.0    |
|2025-03-01 01:00:00|GR     |Fossil Brown coal/Lignite|B02     |501.0    |
|2025-03-01 02:00:00|GR     |Fossil Brown coal/Lignite|B02     |504.0    |
|2025-03-01 03:00:00|GR     |Fossil Brown coal/Lignite|B02     |508.0    |
|2025-03-01 04:00:00|GR     |Fossil Brown coal/Lignite|B02     |506.0    |
+-------------------+-------+-------------------------+--------+---------+
only showing top 5 rows


In [9]:
entsoe_sdf.groupBy('energy_source').agg(F.count('*').alias('rows'), F.sum('actual_MW').alias('total_mw')).orderBy(F.desc('total_mw')).show(truncate=False)

[Stage 1:==============>                                            (1 + 3) / 4]

+-------------------------+----+--------+
|energy_source            |rows|total_mw|
+-------------------------+----+--------+
|Fossil Gas               |143 |352524.0|
|Wind Onshore             |142 |219940.0|
|Solar                    |96  |199580.0|
|Fossil Brown coal/Lignite|124 |50214.0 |
|Hydro Water Reservoir    |127 |25206.0 |
|Hydro Pumped Storage     |53  |13938.0 |
|Fossil Oil               |1   |0.0     |
+-------------------------+----+--------+



In [10]:
era5_wide_pdf = pd.concat([netcdf_to_dataframe(path) for path in era5_files], ignore_index=True)
era5_wide_pdf.head()

,time,lat,lon,ssrd,number,expver,t2m,u100,v100
0,2025-03-01,41.5,19.00,0.0,0,0001,NaN,NaN,NaN
1,2025-03-01,41.5,19.25,0.0,0,0001,NaN,NaN,NaN
2,2025-03-01,41.5,19.50,0.0,0,0001,NaN,NaN,NaN
3,2025-03-01,41.5,19.75,0.0,0,0001,NaN,NaN,NaN
4,2025-03-01,41.5,20.00,0.0,0,0001,NaN,NaN,NaN


In [11]:
id_columns = [col for col in ['time', 'lat', 'lon'] if col in era5_wide_pdf.columns]
value_columns = [col for col in era5_wide_pdf.columns if col not in id_columns]

era5_long_pdf = era5_wide_pdf.melt(
    id_vars=id_columns,
    value_vars=value_columns,
    var_name='variable',
    value_name='value'
)
era5_long_pdf = era5_long_pdf.rename(columns={'time': 'timestamp'})
era5_long_pdf['timestamp'] = pd.to_datetime(era5_long_pdf['timestamp'], errors='coerce').dt.tz_localize(None)
era5_long_pdf['country'] = country
era5_long_pdf = era5_long_pdf.dropna(subset=['timestamp', 'value'])
era5_long_pdf.head()

,timestamp,lat,lon,variable,value,country
0,2025-03-01,41.5,19.00,ssrd,0.0,GR
1,2025-03-01,41.5,19.25,ssrd,0.0,GR
2,2025-03-01,41.5,19.50,ssrd,0.0,GR
3,2025-03-01,41.5,19.75,ssrd,0.0,GR
4,2025-03-01,41.5,20.00,ssrd,0.0,GR


In [12]:
era5_schema = T.StructType([
    T.StructField('timestamp', T.TimestampType(), True),
    T.StructField('lat', T.DoubleType(), True),
    T.StructField('lon', T.DoubleType(), True),
    T.StructField('variable', T.StringType(), True),
    T.StructField('value', T.DoubleType(), True),
    T.StructField('country', T.StringType(), True)
])

era5_records = []
for row in era5_long_pdf[['timestamp', 'lat', 'lon', 'variable', 'value', 'country']].to_dict('records'):
    ts = row['timestamp']
    if hasattr(ts, 'to_pydatetime'):
        ts = ts.to_pydatetime()
    era5_records.append({
        'timestamp': ts,
        'lat': float(row['lat']) if row['lat'] is not None else None,
        'lon': float(row['lon']) if row['lon'] is not None else None,
        'variable': row['variable'],
        'value': float(row['value']) if row['value'] is not None else None,
        'country': row['country'],
    })

era5_sdf = spark.createDataFrame(era5_records, schema=era5_schema)
era5_sdf.printSchema()
era5_sdf.show(5, truncate=False)

root
 |-- timestamp: timestamp (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- variable: string (nullable = true)
 |-- value: double (nullable = true)
 |-- country: string (nullable = true)

+-------------------+----+-----+--------+-----+-------+
|timestamp          |lat |lon  |variable|value|country|
+-------------------+----+-----+--------+-----+-------+
|2025-03-01 00:00:00|41.5|19.0 |ssrd    |0.0  |GR     |
|2025-03-01 00:00:00|41.5|19.25|ssrd    |0.0  |GR     |
|2025-03-01 00:00:00|41.5|19.5 |ssrd    |0.0  |GR     |
|2025-03-01 00:00:00|41.5|19.75|ssrd    |0.0  |GR     |
|2025-03-01 00:00:00|41.5|20.0 |ssrd    |0.0  |GR     |
+-------------------+----+-----+--------+-----+-------+
only showing top 5 rows


26/04/14 10:26:25 WARN TaskSetManager: Stage 4 contains a task of very large size (3612 KiB). The maximum recommended task size is 1000 KiB.


In [ ]:
entsoe_hourly = entsoe_sdf.groupBy('timestamp', 'country').agg(F.sum('actual_MW').alias('generation_mw'))
era5_country = era5_sdf.groupBy('timestamp', 'country', 'variable').agg(F.avg('value').alias('avg_value'))
era5_wide = era5_country.groupBy('timestamp', 'country').pivot('variable').agg(F.first('avg_value'))
joined = entsoe_hourly.join(era5_wide, ['timestamp', 'country'], 'left').orderBy('timestamp')

joined.show(10, truncate=False)

In [ ]:
notebook_out = ROOT / 'silver' / 'notebook'
entsoe_out = notebook_out / 'entsoe_generation_local'
era5_out = notebook_out / 'era5_weather_local'
joined_out = notebook_out / 'generation_weather_join_local'

entsoe_sdf.coalesce(1).write.mode('overwrite').parquet(str(entsoe_out))
era5_sdf.coalesce(1).write.mode('overwrite').parquet(str(era5_out))
joined.coalesce(1).write.mode('overwrite').parquet(str(joined_out))

print(entsoe_out)
print(era5_out)
print(joined_out)

In [15]:
psr_map_expr = F.create_map(*[F.lit(x) for kv in SOURCE_MAP.items() for x in kv])

def relabel_psr_sources(df):
    if 'psr_type' not in df.columns or 'energy_source' not in df.columns:
        return df
    return df.withColumn(
        'energy_source',
        F.coalesce(psr_map_expr[F.col('psr_type')], F.col('energy_source'))
    )

entsoe_check = relabel_psr_sources(spark.read.parquet(str(ROOT / "silver/notebook/entsoe_generation_local")))
era5_check = spark.read.parquet(str(ROOT / "silver/notebook/era5_weather_local"))
join_check = relabel_psr_sources(spark.read.parquet(str(ROOT / "silver/notebook/generation_weather_join_local")))

print("ENTSO-E rows:", entsoe_check.count())
print("ERA5 rows:", era5_check.count())
print("JOIN rows:", join_check.count())

entsoe_check.show(5, truncate=False)
era5_check.show(5, truncate=False)
join_check.show(5, truncate=False)


ENTSO-E rows: 686
ERA5 rows: 285360
JOIN rows: 144
+-------------------+-------+-------------------------+--------+---------+
|timestamp          |country|energy_source            |psr_type|actual_MW|
+-------------------+-------+-------------------------+--------+---------+
|2025-03-01 00:00:00|GR     |Fossil Brown coal/Lignite|B02     |493.0    |
|2025-03-01 01:00:00|GR     |Fossil Brown coal/Lignite|B02     |501.0    |
|2025-03-01 02:00:00|GR     |Fossil Brown coal/Lignite|B02     |504.0    |
|2025-03-01 03:00:00|GR     |Fossil Brown coal/Lignite|B02     |508.0    |
|2025-03-01 04:00:00|GR     |Fossil Brown coal/Lignite|B02     |506.0    |
+-------------------+-------+-------------------------+--------+---------+
only showing top 5 rows
+-------------------+----+-----+--------+-----+-------+
|timestamp          |lat |lon  |variable|value|country|
+-------------------+----+-----+--------+-----+-------+
|2025-03-01 00:00:00|41.5|19.0 |ssrd    |0.0  |GR     |
|2025-03-01 00:00:00|41.5|

In [16]:
print('joined partitions before repartition:', joined.rdd.getNumPartitions())
joined_repart = joined.repartition(4)
print('joined partitions after repartition:', joined_repart.rdd.getNumPartitions())

26/04/14 10:26:40 WARN TaskSetManager: Stage 54 contains a task of very large size (3612 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

joined partitions before repartition: 1


26/04/14 10:26:41 WARN TaskSetManager: Stage 65 contains a task of very large size (3612 KiB). The maximum recommended task size is 1000 KiB.
[Stage 65:=============================>                            (2 + 2) / 4]

joined partitions after repartition: 4


In [ ]:
joined_four_out = notebook_out / 'generation_weather_join_local_4part'
joined_repart.write.mode('overwrite').parquet(str(joined_four_out))
print(joined_four_out)

In [18]:
psr_map_expr = F.create_map(*[F.lit(x) for kv in SOURCE_MAP.items() for x in kv])

def relabel_psr_sources(df):
    if 'psr_type' not in df.columns or 'energy_source' not in df.columns:
        return df
    return df.withColumn(
        'energy_source',
        F.coalesce(psr_map_expr[F.col('psr_type')], F.col('energy_source'))
    )

entsoe_check = relabel_psr_sources(spark.read.parquet(str(entsoe_out)))
era5_check = spark.read.parquet(str(era5_out))
join_check = relabel_psr_sources(spark.read.parquet(str(joined_out)))
join_check_4part = relabel_psr_sources(spark.read.parquet(str(joined_four_out)))

print('ENTSO-E rows:', entsoe_check.count())
print('ERA5 rows:', era5_check.count())
print('JOIN rows:', join_check.count())
print('JOIN rows (4 partitions):', join_check_4part.count())


ENTSO-E rows: 686
ERA5 rows: 285360
JOIN rows: 144
JOIN rows (4 partitions): 144


In [ ]:
entsoe_check.createOrReplaceTempView('entsoe_generation')
era5_check.createOrReplaceTempView('era5_weather')
join_check.createOrReplaceTempView('generation_weather_join')

## Question 1: How many ENTSO-E generation rows were written?

In [20]:
entsoe_check.count()

686

## Question 2: Which energy sources have the largest total generation?

In [21]:
entsoe_check.groupBy('energy_source') \
    .agg(F.sum('actual_MW').alias('total_generation_mw')) \
    .orderBy(F.desc('total_generation_mw')) \
    .show(truncate=False)

+-------------------------+-------------------+
|energy_source            |total_generation_mw|
+-------------------------+-------------------+
|Fossil Gas               |352524.0           |
|Wind Onshore             |219940.0           |
|Solar                    |199580.0           |
|Fossil Brown coal/Lignite|50214.0            |
|Hydro Water Reservoir    |25206.0            |
|Hydro Pumped Storage     |13938.0            |
|Fossil Oil               |0.0                |
+-------------------------+-------------------+



In [22]:
spark.sql("""
SELECT
    energy_source,
    ROUND(SUM(actual_MW), 2) AS total_generation_mw
FROM entsoe_generation
GROUP BY 1
ORDER BY 2 DESC
""").show(truncate=False)

+-------------------------+-------------------+
|energy_source            |total_generation_mw|
+-------------------------+-------------------+
|Fossil Gas               |352524.0           |
|Wind Onshore             |219940.0           |
|Solar                    |199580.0           |
|Fossil Brown coal/Lignite|50214.0            |
|Hydro Water Reservoir    |25206.0            |
|Hydro Pumped Storage     |13938.0            |
|Fossil Oil               |0.0                |
+-------------------------+-------------------+



## Question 3: How many weather rows do we have per variable?

In [23]:
spark.sql("""
SELECT
    variable,
    COUNT(*) AS row_count
FROM era5_weather
GROUP BY 1
ORDER BY 2 DESC
""").show(truncate=False)

+--------+---------+
|variable|row_count|
+--------+---------+
|expver  |85608    |
|number  |85608    |
|ssrd    |28536    |
|v100    |28536    |
|u100    |28536    |
|t2m     |28536    |
+--------+---------+



## Question 4: What are the highest-generation hours in the joined dataset?

In [24]:
join_check.orderBy(F.desc('generation_mw')).select('timestamp', 'country', 'generation_mw').show(10, truncate=False)

+-------------------+-------+-------------+
|timestamp          |country|generation_mw|
+-------------------+-------+-------------+
|2025-03-04 08:00:00|GR     |8639.0       |
|2025-03-05 08:00:00|GR     |8554.0       |
|2025-03-04 10:00:00|GR     |8522.0       |
|2025-03-05 10:00:00|GR     |8452.0       |
|2025-03-05 07:00:00|GR     |8420.0       |
|2025-03-04 18:00:00|GR     |8371.0       |
|2025-03-04 17:00:00|GR     |8306.0       |
|2025-03-04 07:00:00|GR     |8280.0       |
|2025-03-04 09:00:00|GR     |8277.0       |
|2025-03-04 19:00:00|GR     |8232.0       |
+-------------------+-------+-------------+
only showing top 10 rows


In [25]:
spark.sql("""
SELECT
    timestamp,
    country,
    generation_mw
FROM generation_weather_join
ORDER BY generation_mw DESC
LIMIT 10
""").show(truncate=False)

+-------------------+-------+-------------+
|timestamp          |country|generation_mw|
+-------------------+-------+-------------+
|2025-03-04 08:00:00|GR     |8639.0       |
|2025-03-05 08:00:00|GR     |8554.0       |
|2025-03-04 10:00:00|GR     |8522.0       |
|2025-03-05 10:00:00|GR     |8452.0       |
|2025-03-05 07:00:00|GR     |8420.0       |
|2025-03-04 18:00:00|GR     |8371.0       |
|2025-03-04 17:00:00|GR     |8306.0       |
|2025-03-04 07:00:00|GR     |8280.0       |
|2025-03-04 09:00:00|GR     |8277.0       |
|2025-03-04 19:00:00|GR     |8232.0       |
+-------------------+-------+-------------+



## Question 5: What parquet files were created?

In [ ]:
sorted(str(p) for p in notebook_out.glob('*/*.parquet'))

In [27]:
# Optional cleanup when you are completely done with the notebook:
# spark.stop()
